In [15]:
# Import des differents modules
from pathlib import Path

import plotly.express as px
import polars as pl


In [16]:
# Configuration
data_dir = Path("data")

In [17]:
# Lecture des données
# On concatène tous les CSV en un dataframe unique
df = pl.concat(pl.read_csv(csv_path) for csv_path in data_dir.glob("*.csv"))
df.describe()

statistic,turbine_name,timestamp,wind_speed,wind_direction,air_temperature,nacelle_direction,active_power,pitch_angle
str,str,str,f64,f64,f64,f64,f64,f64
"""count""","""1057968""","""1057968""",1.057968e6,1.057968e6,1.057968e6,1.057968e6,1.057968e6,1.057968e6
"""null_count""","""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",null,null,5.374529,178.975028,12.030601,180.682384,335.08536,13.414972
"""std""",null,null,2.565043,95.123872,7.883357,95.574808,431.123112,27.689836
"""min""","""R80711""","""2012-12-31 23:00:00+00:00""",0.0,0.0,-273.20001,0.0,-18.49,-121.26
"""25%""",null,null,3.9400001,93.64,6.0300002,95.559998,8.6999998,-0.99
"""50%""",null,null,5.39,196.74001,11.63,199.58,168.83,-0.97
"""75%""",null,null,6.7199998,247.75999,17.42,250.24,477.29999,4.5
"""max""","""R80790""","""2018-01-12 23:00:00+00:00""",24.27,360.0,70.980003,360.0,2051.8701,262.60999


In [18]:
# Conversion des données de la colonne `timestamp` du type `str` vers `datetime`.
# Corrige aussi le fuseau horaire de `UTC` vers `Europe/Paris`
df = df.with_columns(
    timestamp=pl.col("timestamp")
    .str.to_datetime("%Y-%m-%d %H:%M:%S+00:00")
    .dt.convert_time_zone("Europe/Paris")
)
df.describe()

statistic,turbine_name,timestamp,wind_speed,wind_direction,air_temperature,nacelle_direction,active_power,pitch_angle
str,str,str,f64,f64,f64,f64,f64,f64
"""count""","""1057968""","""1057968""",1.057968e6,1.057968e6,1.057968e6,1.057968e6,1.057968e6,1.057968e6
"""null_count""","""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",null,"""2015-07-10 04:50:30.012249+02:…",5.374529,178.975028,12.030601,180.682384,335.08536,13.414972
"""std""",null,null,2.565043,95.123872,7.883357,95.574808,431.123112,27.689836
"""min""","""R80711""","""2013-01-01 00:00:00+01:00""",0.0,0.0,-273.20001,0.0,-18.49,-121.26
"""25%""",null,"""2014-04-07 04:40:00+02:00""",3.9400001,93.64,6.0300002,95.559998,8.6999998,-0.99
"""50%""",null,"""2015-07-10 09:10:00+02:00""",5.39,196.74001,11.63,199.58,168.83,-0.97
"""75%""",null,"""2016-10-11 13:30:00+02:00""",6.7199998,247.75999,17.42,250.24,477.29999,4.5
"""max""","""R80790""","""2018-01-13 00:00:00+01:00""",24.27,360.0,70.980003,360.0,2051.8701,262.60999


In [19]:
# Graphe de la courbe de puissance pour l'éolienne R80711 en 2014
year = 2014
turbine_name = "R80711"

px.scatter(
    df.filter(
        pl.col("timestamp").dt.year() == year,
        pl.col("turbine_name") == turbine_name,
    ),
    x="wind_speed",
    y="active_power",
    title=f"Active power (kW) vs wind speed (m.s⁻¹) for wind turbine {turbine_name}",
)

In [20]:
# Graphe de la production mensuelle pour l'éolienne R80790
turbine_name = "R80790"

px.bar(
    df.filter(pl.col("turbine_name") == turbine_name)
    .with_columns(
        pl.col("active_power").mul(1 / 6).alias("energy")
    )  # power (kW) * 10min = energy (kWh)
    .sort("timestamp")
    .group_by_dynamic("timestamp", every="1mo")
    .agg(pl.col("energy").sum()),
    x="timestamp",
    y="energy",
    title=f"Monthly production (kWh) for wind turbine {turbine_name}",
)

In [21]:
# Rose des vents pour l'éolienne R80721
wind_turbine = "R80721"
bucket_size = 15
angles = list(range(0, 360, bucket_size))

bucket_counts = (
    pl.DataFrame({"bucket_center": angles})
    .join(
        df.filter(
            (pl.col("turbine_name") == wind_turbine)
            & pl.col("wind_direction").is_not_null()
        )
        .with_columns(
            bucket_center=((pl.col("wind_direction") + bucket_size / 2) % 360).cast(
                pl.Int16
            )
        )
        .group_by("bucket_center")
        .agg(pl.len().alias("count")),
        on="bucket_center",
        how="left",
    )
    .with_columns(pl.col("count").fill_null(0))
)

fig = px.bar_polar(
    bucket_counts,
    r="count",
    theta="bucket_center",
    title=f"Wind direction distribution for wind turbine {wind_turbine}",
)
fig.update_traces(width=bucket_size)
fig.update_layout(
    polar_angularaxis=dict(
        tickmode="array",
        tickvals=angles,
        ticktext=[f"{angle}°" for angle in angles],
        direction="clockwise",
        rotation=0,
    )
)
fig